# GalopTrack — Migration Drive → Cloudflare R2 (à lancer une seule fois)

Copie tes fichiers actuels du Drive vers R2, sous `raw/`, **sans rien modifier sur le Drive**.
Ensuite, lance le workflow GitHub **« Pipeline — migration de l'historique Drive »** : il nettoie
les données, les range dans `data/` et affiche un rapport avant/après.

Avant de lancer : dans Colab, ouvre l'icône 🔑 (Secrets) à gauche et ajoute les 4 secrets
`R2_ACCOUNT_ID`, `R2_ACCESS_KEY_ID`, `R2_SECRET_ACCESS_KEY`, `R2_BUCKET` (mêmes valeurs que dans GitHub).

In [ ]:
!pip install boto3 -q
from google.colab import drive, userdata
drive.mount('/content/drive')

import boto3, gzip, os, shutil
DRIVE_DIR = '/content/drive/MyDrive/GalopTrack'

s3 = boto3.client(
    's3',
    endpoint_url=f"https://{userdata.get('R2_ACCOUNT_ID')}.r2.cloudflarestorage.com",
    aws_access_key_id=userdata.get('R2_ACCESS_KEY_ID'),
    aws_secret_access_key=userdata.get('R2_SECRET_ACCESS_KEY'),
    region_name='auto',
)
BUCKET = userdata.get('R2_BUCKET')
print('Connexion R2 OK :', BUCKET, '-', s3.list_objects_v2(Bucket=BUCKET, MaxKeys=1).get('KeyCount', 0), 'objet(s) existant(s)')

In [ ]:
FICHIERS = ['tracking_data', 'troncons_data', 'chevaux_data', 'cotes_data',
            'rapports_data', 'rapports_combines_data']
MODELE = ['race_model_v7.txt', 'iso_calib_v7.pkl', 'backtest_v7.json']

for nom in FICHIERS:
    src = f'{DRIVE_DIR}/{nom}.csv'
    if not os.path.exists(src):
        print(f'⚠️  {nom}.csv absent du Drive — ignoré')
        continue
    tmp = f'/content/{nom}.csv.gz'
    with open(src, 'rb') as f_in, gzip.open(tmp, 'wb', compresslevel=6) as f_out:
        shutil.copyfileobj(f_in, f_out)
    s3.upload_file(tmp, BUCKET, f'raw/{nom}.csv.gz')
    print(f'✅ {nom}.csv  ({os.path.getsize(src)/1e6:.1f} Mo → {os.path.getsize(tmp)/1e6:.1f} Mo compressé)')

for nom in MODELE:
    src = f'{DRIVE_DIR}/model/{nom}'
    if os.path.exists(src):
        s3.upload_file(src, BUCKET, f'raw/model/{nom}')
        print(f'✅ model/{nom}')
    else:
        print(f'⚠️  model/{nom} absent du Drive')

print()
print('Terminé. Lance maintenant le workflow GitHub « Pipeline — migration de l\'historique Drive ».')

## (Optionnel) Lire les données R2 depuis Colab

Après la migration, pour continuer à explorer dans Colab avec les données à jour :

In [ ]:
import io, pandas as pd
def lire_table(nom):  # 'tracking', 'troncons', 'chevaux', 'rapports', 'rapports_combines'
    obj = s3.get_object(Bucket=BUCKET, Key=f'data/{nom}.csv.gz')
    return pd.read_csv(io.BytesIO(obj['Body'].read()), compression='gzip', low_memory=False)

ch = lire_table('chevaux')
print(len(ch), 'lignes —', ch['date'].min(), '→', ch['date'].max())